# The one-sample t-test
The chocolate-bar test from summary numbers, a t-test on Titanic ages, and how often a sample of 25 rejects H0.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

## 0. Teachers' experience: a t statistic step by step
H0: mu = 5 years, H1: mu < 5; n = 25 teachers, sample mean 4, sample SD 2. Plus the 10% check for independence of a sample of 7 days from at least 365.

In [ ]:
x_bar, mu0, s, n = 4, 5, 2, 25
se = s / np.sqrt(n)
t = (x_bar - mu0) / se
p = stats.t.cdf(t, df=n - 1)
print(f"estimated standard error {se:.2f}, t = {t:.2f}, df = {n - 1}, left-tail p = {p:.4f}")
print(f"one-tailed 5% cutoff t = {stats.t.ppf(0.05, n - 1):.3f}; the same tail on the normal curve: {stats.norm.cdf(t):.4f}")
print("10% condition:", 7 <= 0.10 * 365, 0.10 * 365)

## 1. Chocolate bars: only summary statistics

In [2]:
x_bar, mu0, s, n = 49.7, 50, 1.2, 25
t = (x_bar - mu0) / (s / np.sqrt(n))
# two-tailed p-value from the t CDF with n - 1 degrees of freedom
p = 2 * stats.t.cdf(-abs(t), df=n - 1)
print(f"t = {t:.2f}, df = {n - 1}, p = {p:.3f}")
# the same numbers on the normal curve, for comparison
print(f"z-test p would be {2 * stats.norm.cdf(-abs(t)):.3f}")

t = -1.25, df = 24, p = 0.223
z-test p would be 0.211


## 2. Titanic ages: the population and one sample
Train and test files together hold 1309 passengers; 1046 have a known age.

In [3]:
titanic = pd.concat([pd.read_csv("data/titanic_train.csv"), pd.read_csv("data/titanic_test.csv")])
ages = titanic["Age"].dropna().reset_index(drop=True)
print(len(titanic), "passengers,", len(ages), "known ages, true mean", round(ages.mean(), 2))
# one random sample of 25 ages
sample_age = ages.sample(25, random_state=0)
print(f"sample mean {sample_age.mean():.2f}, sample sd {sample_age.std():.2f}")

1309 passengers, 1046 known ages, true mean 29.88
sample mean 27.20, sample sd 11.53


## 3. Check normality with the Shapiro-Wilk test
H0 of this test: the data comes from a normal distribution.

In [4]:
print("Shapiro-Wilk p, sample of 25:", round(stats.shapiro(sample_age).pvalue, 3))
print("Shapiro-Wilk p, all 1046 ages:", f"{stats.shapiro(ages).pvalue:.1e}")

Shapiro-Wilk p, sample of 25: 0.299
Shapiro-Wilk p, all 1046 ages: 5.7e-11


## 4. H0: mu = 40 against H1: mu < 40


In [5]:
res = stats.ttest_1samp(sample_age, popmean=40, alternative="less")
print(f"t = {res.statistic:.2f}, one-tailed p = {res.pvalue:.6f}")
# by hand: same t and p
t_hand = (sample_age.mean() - 40) / (sample_age.std() / np.sqrt(25))
print(f"by hand: t = {t_hand:.2f}, p = {stats.t.cdf(t_hand, df=24):.6f}")
# the default is two-sided
print(f"two-sided p = {stats.ttest_1samp(sample_age, popmean=40).pvalue:.6f}")


t = -5.55, one-tailed p = 0.000005
by hand: t = -5.55, p = 0.000005
two-sided p = 0.000010


## 5. How often does a sample of 25 reject H0? (the power)

In [6]:
# the claim of section 6 (mu0 = 40), and a claim closer to the truth (mu0 = 35)
for mu0 in [40, 35]:
    rejections = [stats.ttest_1samp(ages.sample(25, random_state=i), mu0, alternative="less").pvalue <= 0.05
                  for i in range(4000)]
    print(f"mu0 = {mu0}: share of samples of 25 that reject H0: {np.mean(rejections):.3f}")


mu0 = 40: share of samples of 25 that reject H0: 0.953


mu0 = 35: share of samples of 25 that reject H0: 0.534


## 6. The five lessons: is 9 minutes on average better than 6?


In [7]:
videos = [7, 9, 5, 11, 13]
res = stats.ttest_1samp(videos, popmean=6, alternative="greater")
print(f"t = {res.statistic:.2f}, p = {res.pvalue:.3f}")

t = 2.12, p = 0.051
